SimCLR backbone and texture comparison

Compare ResNet18, ResNet18 + texture, ResNet50 + texture, and ResNet50.
Each configuration has a readable/bad gate and a separate age classifier:
0, 1, 2, and 3 or older. Original label 6 denotes bad, never an age.
All four experiments share fish-level train/validation/test splits. SimCLR
sees training fish only. Select the final epoch by validation macro F1.
Image-only inputs and texture features contain no fish length or weight.

In [ ]:
import copy
import hashlib
import json
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from PIL import Image
from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, balanced_accuracy_score

ROOT_DIR = Path(os.getenv("TROUT_ROOT_DIR", "/home/jlc3q/data/Trout"))
CODE_DIR = Path(os.getenv("TROUT_CODE_DIR", str(ROOT_DIR / "trout_code_new")))
INPUT_CSV = Path(os.getenv("TROUT_TEXTURE_MASTER", str(
    CODE_DIR / "feature_outputs/master_with_texture_features_full.csv")))
if not INPUT_CSV.exists() and "TROUT_TEXTURE_MASTER" not in os.environ:
    legacy_input = ROOT_DIR / "code_new/feature_outputs/master_with_texture_features_full.csv"
    if legacy_input.exists():
        INPUT_CSV = legacy_input
OUTPUT_DIR = CODE_DIR / "model_outputs/age4_two_stage_comparison"
SEED = 100
CLASS_NAMES = ["0", "1", "2", "3 or older"]
LABELS = list(range(4))
QUALITY_NAMES = ["readable", "bad"]
PIPELINE_NAMES = CLASS_NAMES + ["bad"]
SPLIT_LABELS = list(range(5))
BAD_THRESHOLD = 0.5  # Fixed before test evaluation; higher values pass more scales.
TEST_SIZE = 0.20
VAL_SIZE_OF_REMAINDER = 0.20
SIMCLR_EPOCHS = 10
CLASSIFIER_EPOCHS = 10
SIMCLR_BATCH_SIZE = 128
CLASSIFIER_BATCH_SIZE = 64
SIMCLR_LR = 3e-4
CLASSIFIER_LR = 1e-4
TEMPERATURE = 0.2
NUM_WORKERS = 0  # Safe default for HPC Jupyter multiprocessing.
USE_AMP = False  # Optional CUDA speed/memory optimization.
RUN_TRAINING = False  # Set True after checking the data/split audit.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = USE_AMP and device.type == "cuda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("device:", device, "input:", INPUT_CSV, "output:", OUTPUT_DIR)
print("torch:", torch.__version__, "AMP:", AMP_ENABLED)

Input and cohort audit

Run full texture extraction first. Set TROUT_TEXTURE_MASTER to the actual CSV
if prior outputs are elsewhere. The known code_new full CSV is an automatic
fallback; sample CSVs are never used. Compute targets from original label.
Numeric label 6 follows the existing bad convention; confirm ambiguous GT
entries with the experts before interpreting it as a biological age.

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Set INPUT_CSV to the full texture master: {INPUT_CSV}")
master = pd.read_csv(INPUT_CSV)
required = {"scale_id", "fish_key", "path", "label"}
if required - set(master):
    raise ValueError(f"Missing columns: {sorted(required - set(master))}")
if master["scale_id"].isna().any() or master["scale_id"].duplicated().any():
    raise ValueError("Master scale_id must be non-null and unique; audit Excel joins.")
numeric_label = pd.to_numeric(master["label"], errors="coerce")
invalid = master["label"].notna() & ~numeric_label.isin(range(7))
if invalid.any():
    raise ValueError("Expected integer labels 0..6 in the master table.")
texture_prefixes = ("intensity_", "sobel_", "canny_", "edge_component_",
                    "glcm_", "lbp_", "gabor_", "hog_")
texture_cols = [c for c in master if c.startswith(texture_prefixes)]
if not texture_cols:
    raise ValueError("No image-derived texture columns found; run full extraction.")
cohort = master.loc[numeric_label.notna()].copy()
cohort["target"] = numeric_label.loc[cohort.index].astype(int).clip(upper=3)
cohort.loc[numeric_label.loc[cohort.index].eq(6), "target"] = 4
if cohort["fish_key"].isna().any():
    raise ValueError("Labeled scales need fish_key for leakage-free splitting.")
cohort["fish_key"] = cohort["fish_key"].astype(str).str.strip().str.lower()
texture_values = cohort[texture_cols].apply(pd.to_numeric, errors="raise")
texture_values = texture_values.replace([np.inf, -np.inf], np.nan)
available = texture_values.notna().any(axis=1)
path_exists = cohort["path"].map(lambda p: Path(str(p)).is_file())
print("Excluded unlabeled rows:", len(master) - len(cohort))
print("Labeled rows missing image:", int((~path_exists).sum()))
print("Labeled rows missing all texture features:", int((~available).sum()))
cohort[texture_cols] = texture_values
# All models use the same available image/texture cohort for a paired comparison.
cohort = cohort.loc[path_exists & available].sort_values("scale_id").reset_index(drop=True)
if cohort["fish_key"].eq("").any():
    raise ValueError("Empty fish_key detected.")
display(cohort.groupby("target").agg(scales=("scale_id", "size"),
                                    fish=("fish_key", "nunique")))
readable_gt = cohort.loc[cohort["target"].lt(4)]
print("Fish with conflicting readable age GT:",
      int((readable_gt.groupby("fish_key")["target"].nunique() > 1).sum()))

Fixed fish-level splits and preprocessing

Reuse the saved split manifest only for an identical cohort, labels and texture
table. Validation is 20% of the remaining 80%; test is 20% of fish. Seed search
only checks class presence, never model performance. Conflicting GT remains
scale-level; all scales belonging to a fish stay together.

In [ ]:
fingerprint_cols = ["scale_id", "fish_key", "path", "target"] + texture_cols
digest = hashlib.sha256(pd.util.hash_pandas_object(
    cohort[fingerprint_cols], index=False).values.tobytes()).hexdigest()
split_path = OUTPUT_DIR / "split_manifest.csv"
split_meta_path = OUTPUT_DIR / "split_metadata.json"
split_config = {"seed": SEED, "test_size": TEST_SIZE,
                "val_size_of_remainder": VAL_SIZE_OF_REMAINDER,
                "cohort_sha256": digest, "texture_cols": texture_cols}

def make_split(df):
    for seed in range(SEED, SEED + 500):
        outer = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        rem_idx, test_idx = next(outer.split(df, groups=df["fish_key"]))
        remainder = df.iloc[rem_idx]
        inner = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE_OF_REMAINDER,
                                  random_state=seed + 1000)
        train_sub, val_sub = next(inner.split(remainder, groups=remainder["fish_key"]))
        indices = {"train": rem_idx[train_sub], "validation": rem_idx[val_sub],
                   "test": test_idx}
        if all(set(df.iloc[idx]["target"]) == set(SPLIT_LABELS) for idx in indices.values()):
            manifest = df[["scale_id", "fish_key", "target"]].copy()
            manifest["split"] = ""
            for name, idx in indices.items():
                manifest.loc[idx, "split"] = name
            return manifest, seed
    raise ValueError("Cannot put all four ages and bad in each fish split. Inspect GT fish counts.")

if split_path.exists() or split_meta_path.exists():
    if not (split_path.exists() and split_meta_path.exists()):
        raise ValueError("Incomplete saved split; choose a new OUTPUT_DIR.")
    previous = json.loads(split_meta_path.read_text())
    if any(previous.get(k) != v for k, v in split_config.items()):
        raise ValueError("Cohort/config changed. Choose a new OUTPUT_DIR to preserve the old run.")
    manifest = pd.read_csv(split_path)
else:
    manifest, selected_seed = make_split(cohort)
    manifest.to_csv(split_path, index=False)
    split_meta_path.write_text(json.dumps({**split_config, "selected_seed": selected_seed}, indent=2))
if manifest["scale_id"].isna().any() or manifest["scale_id"].duplicated().any():
    raise ValueError("Split manifest needs unique, non-null scale_id values.")
# Preserve the source labeled/unlabeled split; assign model partitions separately.
cohort["model_split"] = cohort["scale_id"].map(manifest.set_index("scale_id")["split"])
parts = {name: cohort.loc[cohort["model_split"].eq(name)].reset_index(drop=True)
         for name in ["train", "validation", "test"]}
fish_sets = {name: set(df["fish_key"]) for name, df in parts.items()}
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert not fish_sets[a] & fish_sets[b], f"Fish leakage: {a}/{b}"
assert len(manifest) == len(cohort) and cohort["model_split"].isin(parts).all()
display(pd.crosstab(cohort["model_split"], cohort["target"]))
print("Fish counts:", {k: len(v) for k, v in fish_sets.items()})
if parts["train"][texture_cols].isna().all().any():
    raise ValueError("A texture column is empty in training; remove it explicitly and restart.")
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
texture_arrays = {"train": scaler.fit_transform(imputer.fit_transform(
    parts["train"][texture_cols])).astype(np.float32)}
for name in ["validation", "test"]:
    texture_arrays[name] = scaler.transform(imputer.transform(
        parts[name][texture_cols])).astype(np.float32)
joblib.dump({"columns": texture_cols, "imputer": imputer, "scaler": scaler},
            OUTPUT_DIR / "texture_preprocessing.joblib")

In [ ]:
normalization = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
simclr_transform = T.Compose([
    T.Resize((224, 224)), T.RandomResizedCrop(224, scale=(0.65, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
    T.ColorJitter(brightness=0.25, contrast=0.25), T.RandomGrayscale(p=0.2),
    T.ToTensor(), normalization,
])
# Match texture to the same unaugmented image; keep this identical for all heads.
classifier_transform = T.Compose([T.Resize((224, 224)), T.ToTensor(), normalization])

class ScaleDataset(Dataset):
    def __init__(self, frame, transform, texture=None, contrastive=False):
        self.frame = frame.reset_index(drop=True)
        self.transform, self.texture, self.contrastive = transform, texture, contrastive

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        with Image.open(self.frame.iloc[idx]["path"]) as image:
            image = image.convert("RGB")
        if self.contrastive:
            return self.transform(image), self.transform(image)
        return (self.transform(image), torch.from_numpy(self.texture[idx]),
                int(self.frame.iloc[idx]["target"]))

def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=NUM_WORKERS,
                      pin_memory=device.type == "cuda",
                      generator=torch.Generator().manual_seed(SEED))

def make_backbone(name, pretrained=True):
    builders = {"resnet18": (models.resnet18, models.ResNet18_Weights.DEFAULT),
                "resnet50": (models.resnet50, models.ResNet50_Weights.DEFAULT)}
    builder, weights = builders[name]
    backbone = builder(weights=weights if pretrained else None)
    feature_dim = backbone.fc.in_features
    backbone.fc = nn.Identity()
    return backbone, feature_dim

def ntxent(z1, z2, temperature=TEMPERATURE):
    n = len(z1)
    if n < 2:
        raise ValueError("SimCLR requires at least two images for negative pairs.")
    z = F.normalize(torch.cat([z1, z2]).float(), dim=1)
    logits = z @ z.T / temperature
    logits = logits.masked_fill(torch.eye(2*n, device=z.device, dtype=torch.bool), -torch.inf)
    positive_indices = (torch.arange(2*n, device=z.device) + n) % (2*n)
    return F.cross_entropy(logits, positive_indices)

def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

class AgeClassifier(nn.Module):
    def __init__(self, backbone, feature_dim, texture_dim=0, n_classes=4):
        super().__init__()
        self.backbone = backbone
        self.texture_branch = (nn.Sequential(nn.Linear(texture_dim, 32), nn.ReLU())
                               if texture_dim else None)
        self.head = nn.Sequential(nn.Linear(feature_dim + (32 if texture_dim else 0), 128),
                                  nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n_classes))

    def forward(self, images, texture):
        image_features = self.backbone(images)
        if self.texture_branch is not None:
            image_features = torch.cat([image_features, self.texture_branch(texture)], dim=1)
        return self.head(image_features)

Train two SimCLR backbones

Image-only and fusion start from the same SimCLR weights for each backbone.
SimCLR includes readable and bad training scales, with no held-out fish.
All classifier experiments fine-tune the full backbone, as in the old notebook.
Use equal batch sizes; if ResNet50 runs out of memory, restart all experiments
in a new OUTPUT_DIR with a smaller common batch size. AMP is optional.

In [ ]:
def pretrain(name):
    set_seed()
    backbone, dim = make_backbone(name)
    backbone = backbone.to(device)
    projector = nn.Sequential(nn.Linear(dim, 128), nn.ReLU(), nn.Linear(128, 128)).to(device)
    optimizer = torch.optim.AdamW(list(backbone.parameters()) + list(projector.parameters()),
                                  lr=SIMCLR_LR, weight_decay=1e-4)
    amp_scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    ds = ScaleDataset(parts["train"], simclr_transform, contrastive=True)
    if len(ds) < SIMCLR_BATCH_SIZE:
        raise ValueError("Reduce the common SimCLR batch size; no full batch available.")
    loader = make_loader(ds, SIMCLR_BATCH_SIZE, shuffle=True, drop_last=True)
    history = []
    for epoch in range(SIMCLR_EPOCHS):
        backbone.train(); projector.train()
        total, count = 0.0, 0
        for x1, x2 in tqdm(loader, desc=f"{name} SimCLR {epoch+1}/{SIMCLR_EPOCHS}"):
            x1, x2 = x1.to(device), x2.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
                loss = ntxent(projector(backbone(x1)), projector(backbone(x2)))
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite SimCLR loss.")
            amp_scaler.scale(loss).backward()
            amp_scaler.step(optimizer); amp_scaler.update()
            total += loss.item() * len(x1); count += len(x1)
        history.append({"epoch": epoch+1, "loss": total/count})
        print(history[-1])
    state = cpu_state(backbone)
    torch.save({"backbone_state_dict": state, "feature_dim": dim,
                "cohort_sha256": digest, "history": history}, OUTPUT_DIR / f"{name}_simclr.pt")
    pd.DataFrame(history).to_csv(OUTPUT_DIR / f"{name}_pretraining_history.csv", index=False)
    return state, dim

if RUN_TRAINING:
    pretrained_states = {name: pretrain(name) for name in ["resnet18", "resnet50"]}
else:
    print("Audit the cohort above, then set RUN_TRAINING=True and run from the setup cell.")

In [ ]:
@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    truth, probabilities = [], []
    for images, texture, labels in loader:
        with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
            logits = model(images.to(device), texture.to(device))
        probabilities.append(logits.float().softmax(1).cpu().numpy())
        truth.extend(labels.numpy().tolist())
    probs = np.concatenate(probabilities)
    return np.asarray(truth), probs.argmax(1), probs

def metrics_for(truth, pred, names=CLASS_NAMES):
    report = classification_report(truth, pred, labels=list(range(len(names))), target_names=names,
                                   output_dict=True, zero_division=0)
    return {"accuracy": accuracy_score(truth, pred),
            "balanced_accuracy": balanced_accuracy_score(truth, pred),
            "macro_f1": report["macro avg"]["f1-score"],
            "weighted_f1": report["weighted avg"]["f1-score"],
            "support": len(truth)}, report

def task_data(task):
    frames, arrays = {}, {}
    for split, frame in parts.items():
        mask = frame["target"].lt(4).to_numpy() if task == "age" else np.ones(len(frame), dtype=bool)
        frames[split] = frame.loc[mask].copy().reset_index(drop=True)
        arrays[split] = texture_arrays[split][mask]
        if task == "quality":
            frames[split]["target"] = frames[split]["target"].eq(4).astype(int)
    return frames, arrays

def train_classifier(name, backbone_name, fusion, task="age"):
    set_seed()
    names = CLASS_NAMES if task == "age" else QUALITY_NAMES
    labels = list(range(len(names)))
    task_parts, task_texture = task_data(task)
    backbone, dim = make_backbone(backbone_name, pretrained=False)
    backbone.load_state_dict(pretrained_states[backbone_name][0], strict=True)
    model = AgeClassifier(backbone, dim, len(texture_cols) if fusion else 0, len(names)).to(device)
    loaders = {split: make_loader(ScaleDataset(frame, classifier_transform, task_texture[split]),
                                 CLASSIFIER_BATCH_SIZE, shuffle=split == "train")
               for split, frame in task_parts.items()}
    counts = task_parts["train"]["target"].value_counts().reindex(labels, fill_value=0)
    if counts.eq(0).any():
        raise ValueError(f"Missing {task} class in training.")
    weights = torch.tensor((counts.sum()/(len(names)*counts)).to_numpy(), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CLASSIFIER_LR, weight_decay=1e-4)
    amp_scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    best_score, best_state, best_epoch = -1.0, None, None
    history = []
    for epoch in range(CLASSIFIER_EPOCHS):
        model.train()
        total, count = 0.0, 0
        for images, texture, y in tqdm(loaders["train"], desc=f"{name} {epoch+1}/{CLASSIFIER_EPOCHS}"):
            images, texture, y = images.to(device), texture.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=AMP_ENABLED):
                loss = criterion(model(images, texture), y)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite classifier loss.")
            amp_scaler.scale(loss).backward()
            amp_scaler.step(optimizer); amp_scaler.update()
            total += loss.item()*len(y); count += len(y)
        truth, pred, _ = predict_model(model, loaders["validation"])
        val_metrics, _ = metrics_for(truth, pred, names)
        history.append({"epoch": epoch+1, "train_loss": total/count,
                        **{f"val_{k}": v for k,v in val_metrics.items()}})
        print(history[-1])
        if val_metrics["macro_f1"] > best_score:
            best_score, best_epoch = val_metrics["macro_f1"], epoch+1
            best_state = cpu_state(model)
    model.load_state_dict(best_state, strict=True)
    torch.save({"model_state_dict": best_state, "backbone": backbone_name,
                "feature_dim": dim, "texture_cols": texture_cols if fusion else [],
                "class_names": names, "task": task, "best_epoch": best_epoch,
                "validation_macro_f1": best_score, "cohort_sha256": digest},
               OUTPUT_DIR / f"{name}_best.pt")
    pd.DataFrame(history).to_csv(OUTPUT_DIR / f"{name}_training_history.csv", index=False)
    truth, pred, probs = predict_model(model, loaders["test"])
    metrics, report = metrics_for(truth, pred, names)
    print(classification_report(truth, pred, labels=labels, target_names=names,
                                digits=4, zero_division=0))
    cm = pd.DataFrame(confusion_matrix(truth, pred, labels=labels),
                      index=names, columns=names)
    cm.to_csv(OUTPUT_DIR / f"{name}_confusion_matrix.csv")
    display(cm)
    (OUTPUT_DIR / f"{name}_classification_report.json").write_text(json.dumps(report, indent=2))
    predictions = task_parts["test"][["scale_id", "fish_key", "target"]].copy()
    predictions["prediction"] = pred
    for k in labels:
        predictions[f"prob_{k}"] = probs[:,k]
    predictions.to_csv(OUTPUT_DIR / f"{name}_test_predictions.csv", index=False)
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {"model": name, **metrics, "best_epoch": best_epoch,
            "validation_macro_f1": best_score}

Paired comparison

Adding bad changes the fish cohort and split relative to the previous notebook.
Old checkpoints cannot be reused: a former training fish might now be in test.
Compare the four rows below, not historical scores on another split.
Fusion has a small extra texture branch, so gains may also reflect capacity.
Test predictions are retained for fish-level uncertainty analysis later.

In [ ]:
experiments = [
    ("simclr_resnet18", "resnet18", False),
    ("simclr_resnet18_texture", "resnet18", True),
    ("simclr_resnet50_texture", "resnet50", True),
    ("simclr_resnet50", "resnet50", False),
]
if RUN_TRAINING:
    results, quality_results = [], []
    for name, backbone_name, fusion in experiments:
        results.append(train_classifier(name, backbone_name, fusion))
        quality_results.append(train_classifier(name + "_quality", backbone_name, fusion, task="quality"))
        results_df = pd.DataFrame(results)
        results_df.to_csv(OUTPUT_DIR / "comparison_results.csv", index=False)
        quality_results_df = pd.DataFrame(quality_results)
        quality_results_df.to_csv(OUTPUT_DIR / "quality_comparison_results.csv", index=False)
    display(results_df.sort_values("balanced_accuracy", ascending=False))
    display(quality_results_df.sort_values("balanced_accuracy", ascending=False))
    config = {"seed": SEED, "simclr_epochs": SIMCLR_EPOCHS,
              "classifier_epochs": CLASSIFIER_EPOCHS, "simclr_batch_size": SIMCLR_BATCH_SIZE,
              "classifier_batch_size": CLASSIFIER_BATCH_SIZE, "temperature": TEMPERATURE,
              "simclr_lr": SIMCLR_LR, "classifier_lr": CLASSIFIER_LR,
              "use_amp": AMP_ENABLED, "torch_version": torch.__version__,
              "class_names": CLASS_NAMES, "uses_length_weight": False,
              "quality_class_names": QUALITY_NAMES, "bad_threshold": BAD_THRESHOLD,
              "simclr_uses_test_or_validation_fish": False, "cohort_sha256": digest}
    (OUTPUT_DIR / "run_config.json").write_text(json.dumps(config, indent=2))
else:
    print("Training disabled. Enable RUN_TRAINING in the setup cell when ready.")

Two-stage evaluation: readable/bad -> age

Evaluate each matched quality/age model pair on ALL labeled test scales.
A bad prediction stops the pipeline. Only scales passed by the gate are sent
to the age model. The five final outcomes are 0, 1, 2, 3 or older, and bad.
The age-only table above uses GT-readable scales (an oracle gate); it does not
measure deployment performance. The end-to-end table counts rejected readable
scales and passed bad scales as errors. Report age accuracy among accepted
GT-readable scales with coverage, never alone. The bad threshold is fixed at
0.5; choose any alternative threshold using validation only, before testing.

In [ ]:
def load_task_model(name, backbone_name, fusion, n_classes):
    checkpoint = torch.load(OUTPUT_DIR / f"{name}_best.pt", map_location="cpu", weights_only=True)
    if checkpoint["cohort_sha256"] != digest:
        raise ValueError("Checkpoint belongs to another cohort/split.")
    backbone, dim = make_backbone(backbone_name, pretrained=False)
    model = AgeClassifier(backbone, dim, len(texture_cols) if fusion else 0, n_classes)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model.to(device)

def pipeline_statistics(truth, final_pred):
    readable = truth < 4
    bad = ~readable
    accepted = final_pred < 4
    accepted_readable = readable & accepted
    metrics, _ = metrics_for(truth, final_pred, PIPELINE_NAMES)
    return {**metrics,
            "readable_count": int(readable.sum()), "bad_count": int(bad.sum()),
            "readable_rejection_rate": float((~accepted[readable]).mean()),
            "bad_pass_rate": float(accepted[bad].mean()),
            "readable_coverage": float(accepted[readable].mean()),
            "readable_end_to_end_accuracy": float((final_pred[readable] == truth[readable]).mean()),
            "accepted_readable_age_accuracy": (float((final_pred[accepted_readable] == truth[accepted_readable]).mean())
                                                if accepted_readable.any() else np.nan)}

def evaluate_pipeline(name, backbone_name, fusion):
    test = parts["test"]
    texture = texture_arrays["test"]
    quality_model = load_task_model(name + "_quality", backbone_name, fusion, 2)
    quality_frame = test.copy()
    quality_frame["target"] = quality_frame["target"].eq(4).astype(int)
    loader = make_loader(ScaleDataset(quality_frame, classifier_transform, texture), CLASSIFIER_BATCH_SIZE)
    _, _, quality_probs = predict_model(quality_model, loader)
    passed = quality_probs[:, 1] < BAD_THRESHOLD
    final_pred = np.full(len(test), 4, dtype=int)
    age_probs = np.full((len(test), 4), np.nan)
    del quality_model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    if passed.any():
        age_model = load_task_model(name, backbone_name, fusion, 4)
        age_frame = test.loc[passed].reset_index(drop=True)
        loader = make_loader(ScaleDataset(age_frame, classifier_transform, texture[passed]), CLASSIFIER_BATCH_SIZE)
        _, age_pred, probs = predict_model(age_model, loader)
        final_pred[passed] = age_pred
        age_probs[passed] = probs
        del age_model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    truth = test["target"].to_numpy()
    row = {"model": name, **pipeline_statistics(truth, final_pred)}
    predictions = test[["scale_id", "fish_key", "target"]].copy()
    predictions["bad_probability"] = quality_probs[:, 1]
    predictions["gate_passed"] = passed
    predictions["final_prediction"] = final_pred
    for k in LABELS:
        predictions[f"age_prob_{k}"] = age_probs[:, k]
    predictions.to_csv(OUTPUT_DIR / f"{name}_pipeline_predictions.csv", index=False)
    cm = pd.DataFrame(confusion_matrix(truth, final_pred, labels=SPLIT_LABELS),
                      index=PIPELINE_NAMES, columns=PIPELINE_NAMES)
    cm.to_csv(OUTPUT_DIR / f"{name}_pipeline_confusion_matrix.csv")
    print(name)
    print(classification_report(truth, final_pred, labels=SPLIT_LABELS,
                                target_names=PIPELINE_NAMES, digits=4, zero_division=0))
    display(cm)
    return row

if RUN_TRAINING:
    pipeline_results = []
    for name, backbone_name, fusion in experiments:
        pipeline_results.append(evaluate_pipeline(name, backbone_name, fusion))
        pipeline_results_df = pd.DataFrame(pipeline_results)
        pipeline_results_df.to_csv(OUTPUT_DIR / "pipeline_comparison_results.csv", index=False)
    display(pipeline_results_df)
else:
    print("Two-stage evaluation runs after training the four model pairs.")